# 10 — Prensa online MediaCloud: Inmigración

**Objetivo:** construir un dataset limpio de noticias sobre inmigración y extraer el cuerpo completo de cada noticia a partir de las URLs de MediaCloud.

En este notebook **no hacemos NLP todavía**. Primero cerramos la capa de datos:

1. Cargar CSV de MediaCloud.
2. Limpiar columnas y fechas.
3. Revisar duplicados.
4. Probar extracción de texto con Trafilatura.
5. Medir tasa de éxito.
6. Extraer cuerpos de noticias.
7. Guardar dataset procesado.


## 1. Instalación e importación de librerías

In [3]:
import pandas as pd
import numpy as np
import trafilatura
from tqdm import tqdm
from pathlib import Path
import time

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)

## 2. Definir rutas

Cambia estas rutas según dónde guardes el CSV descargado de MediaCloud.

In [7]:
# Ruta del archivo RAW de MediaCloud
RUTA_RAW = Path(
    r"C:\Users\herre\OneDrive\Desktop\IT ACADEMY\ESPECIALIDAD\EJERCICICIOS\SPRINT 13\PRENSA\RAW\2023-inmigracion-query2.csv"
)

# Carpeta donde guardaremos el archivo con el texto extraído
RUTA_PROCESADOS = Path(
    r'C:\Users\herre\OneDrive\Desktop\IT ACADEMY\ESPECIALIDAD\EJERCICICIOS\SPRINT 13\PRENSA\PROCESADOS'
)
RUTA_PROCESADOS.mkdir(parents=True, exist_ok=True)

RUTA_SALIDA = RUTA_PROCESADOS / 'inmigracion_2023_con_texto.csv'

## 3. Cargar CSV de MediaCloud

In [8]:
df = pd.read_csv(RUTA_RAW)

print(df.shape)
print(df.columns.tolist())

(134480, 8)
['id', 'indexed_date', 'language', 'media_name', 'media_url', 'publish_date', 'title', 'url']


## 4. Revisión inicial de columnas

In [9]:
df.columns

Index(['id', 'indexed_date', 'language', 'media_name', 'media_url',
       'publish_date', 'title', 'url'],
      dtype='object')

## 5. Limpieza básica

Nos quedamos con las columnas útiles: fecha, medio, título y URL.

In [10]:
columnas_utiles = ['id', 'publish_date', 'media_name', 'media_url', 'title', 'url', 'language']

columnas_existentes = [col for col in columnas_utiles if col in df.columns]
df = df[columnas_existentes].copy()

# Convertir fecha
df['publish_date'] = pd.to_datetime(df['publish_date'], errors='coerce')

# Crear año y mes
df['anio'] = df['publish_date'].dt.year
df['mes'] = df['publish_date'].dt.month

# Limpiar textos básicos
df['title'] = df['title'].astype(str).str.strip()
df['url'] = df['url'].astype(str).str.strip()
df['media_name'] = df['media_name'].astype(str).str.strip()

df.head()

,id,publish_date,media_name,media_url,title,url,language,anio,mes
0,2a65389eca68767026ff38957369cc459475bfcac8ec4593a203ac0af9de9e8f,2023-07-28,laregion.es,laregion.es,Normas y recomendaciones de seguridad en zonas de baño,https://www.laregion.es/sociedad/normas-recomendaciones-seguridad-zonas-bano_1_20230728-2395889.html,es,2023,7
1,5bdf7fb1b15502cec53f099f25c7834b3566f3ec443582e53140ae7a07eb8222,2023-04-21,rtve.es,rtve.es,Todas estas películas han ganado en los Premios Platino y pueden verse gratis en RTVE Play,https://www.rtve.es/television/20230421/peliculas-cine-habla-hispana-espanol-iberoamericanas-gratis-ganadoras-premio...,es,2023,4
2,2caeb828237c67170aaa4749247049829d804e7c3d9fca57083182679f173389,2023-10-05,ondacero.es,ondacero.es,El Gabinete: Objetivos y trascendencia de la cumbre de los líderes europeos en Granada,https://www.ondacero.es/podcast/programas/julia-en-la-onda/gabinete/gabinete-objetivos-trascendencia-cumbre-lideres-...,es,2023,10
3,af9cbd7f85668c487799439ee59cf27aad052cee107360dcf86e761609520e5c,2023-11-20,ondacero.es,ondacero.es,El síndrome de Ulises,https://www.ondacero.es/podcast/programas/rosa-vientos/big-bang-mado/sindrome-ulises_20231120655ad1442670b0e4c813e4e...,es,2023,11
4,56fcea76e0bee47015345e47ff134172011294f091c9c1b06483518334b9ec33,2023-06-20,rtve.es,rtve.es,"El icónico coche de 'Gran Torino', la comunidad Hmong y otras curiosidades del cascarrabias Kowalski",https://www.rtve.es/television/20230620/gran-torino-curiosidades-coche-etnia-hmong/2449977.shtml,es,2023,6


## 6. Diagnóstico rápido del dataset

In [11]:
print('Noticias totales:', len(df))
print('URLs únicas:', df['url'].nunique())
print('Medios únicos:', df['media_name'].nunique())
print('Fechas mín / máx:', df['publish_date'].min(), '→', df['publish_date'].max())

df['anio'].value_counts().sort_index()

Noticias totales: 134480
URLs únicas: 134480
Medios únicos: 234
Fechas mín / máx: 2023-01-01 00:00:00 → 2023-12-31 00:00:00


anio
2023    134480
Name: count, dtype: int64

## 7. Eliminar duplicados

MediaCloud puede devolver la misma URL más de una vez. Para extraer texto, no nos interesa repetir URLs.

In [12]:
df = df.drop_duplicates(subset='url').copy()
df = df.dropna(subset=['url', 'publish_date', 'title'])

print('Filas después de eliminar duplicados:', len(df))

Filas después de eliminar duplicados: 134480


## 8. Comprobar principales medios

In [13]:
df['media_name'].value_counts().head(20)

media_name
cope.es                        5902
elperiodico.com                5669
elpais.com                     3807
elnacional.cat                 3433
abc.es                         3287
europapress.es                 2939
eldia.es                       2824
canarias7.es                   2623
publico.es                     2533
levante-emv.com                2525
lavanguardia.com               2485
eldiario.es                    2441
diariodemallorca.es            2282
lne.es                         2264
farodevigo.es                  2139
diariodeibiza.es               1997
ara.cat                        1942
elperiodicomediterraneo.com    1916
marca.com                      1870
laopiniondemurcia.es           1766
Name: count, dtype: int64

## 9. Validación temática de la búsqueda

In [14]:
# 9. Muestra aleatoria de titulares

df['title'].sample(20, random_state=42).tolist()

['El joven acusado de patronear una patera de Argelia a Cabrera dice que era un pasajero más y que pagó 1.700 euros',
 'La poliomielitis y el sarampión vuelven a resurgir: los pediatras piden no bajar la cobertura vacunal',
 'Rueda recrimina al Gobierno la información "insuficiente" sobre la llegada de migrantes y ve descoordinación',
 'Albert Galimany, el músic del Penedès que va triomfar al Panamà',
 'Galiverso somerxido: asÃ\xad se recrea unha visita virtual polo fondo mariÃ±o de Galicia',
 '¿Por qué no denuncian las mujeres?',
 'La salud de Mohamed VI, el secreto mejor guardado de Marruecos',
 'Un important desplegament policial dissuadeix la manifestació okupa pel Kubo i la Ruïna a Barcelona',
 'Rubiales dimite por lo penal',
 'Clavijo, uno de los pocos presidentes autonómicos que se congela el sueldo en 2024',
 'Falsos culpables en un país de quijotes: cómo unos ciudadanos lucharon por dos condenados en Cataluña',
 'Varios okupas abandonan voluntariamente un piso de la calle Mate

## 10. Filtrado temático preliminar

In [17]:
terminos = [
    'inmigr',
    'migrante',
    'migración',
    'refugiado',
    'asilo',
    'patera'
]

patron = '|'.join(terminos)

df_filtrado = df[
    df['title'].str.lower().str.contains(
        patron,
        na=False
    )
].copy()

print("Noticias originales:", len(df))
print("Noticias filtradas:", len(df_filtrado))

Noticias originales: 134480
Noticias filtradas: 22273


In [18]:
df_filtrado['title'].sample(20, random_state=42).tolist()

['La Guardia Civil detiene a nueve personas por favorecer el empadronamiento fraudulento de migrantes en situación irregular',
 'La llegada de migrantes a la frontera de EE UU con México, en imágenes',
 'Rescatan una patera con 85 migrantes y una persona fallecida en Gran Canaria',
 'Mueren diez migrantes tras hundirse una embarcación frente a las costas de Túnez',
 'Siete horas de angustia y espera para un rescate que deja trece muertos más en la ruta canaria de la inmigración',
 'Manuel Domínguez sostiene que la política en materia de inmigración “debe tener en Canarias el eje central de sus actuaciones”',
 'Desarticulada una mafia que vendía documentación falsa a los migrantes para salir de Canarias',
 'Tres fábricas ilegales de tabaco explotaban a refugiados ucranianos',
 'Fallece un migrante en un cayuco interceptado en Tenerife',
 'Al menos cuatro muertos y 51 desaparecidos tras el naufragio de un barco de migrantes en Túnez',
 'La falta de medios policiales impide investigar a l

In [19]:
print(df_filtrado.shape)

(22273, 9)


## 11. Función para extraer texto con Trafilatura

In [20]:
def extraer_texto_trafilatura(url, pausa=0.5):

    try:
        descargado = trafilatura.fetch_url(url)
        time.sleep(pausa)

        if descargado is None:
            return None

        texto = trafilatura.extract(
            descargado,
            include_comments=False,
            include_tables=False,
            favor_precision=True
        )

        if texto is None:
            return None

        texto = texto.strip()

        if len(texto) < 300:
            return None

        return texto

    except Exception as e:
        return None

In [21]:
df_test = df_filtrado.sample(
    100,
    random_state=42
).copy()

print(len(df_test))

100


In [22]:
df_test["texto"] = df_test["url"].apply(
    extraer_texto_trafilatura
)

In [23]:
exito = (
    df_test["texto"]
    .notna()
    .mean()
    * 100
)

print(f"Tasa de éxito: {exito:.2f}%")

Tasa de éxito: 86.00%


## 10. Prueba piloto con 100 noticias

Antes de extraer miles de noticias, medimos la tasa de éxito.

In [24]:
df_test = df.sample(100, random_state=42).copy()

tqdm.pandas()
df_test['texto'] = df_test['url'].progress_apply(extraer_texto_trafilatura)

exito = df_test['texto'].notna().mean() * 100
print(f'Tasa de extracción: {exito:.2f}%')

df_test[['media_name', 'title', 'url', 'texto']].head()

100%|██████████| 100/100 [01:56<00:00,  1.16s/it]

Tasa de extracción: 91.00%


,media_name,title,url,texto
12387,elconfidencialdigital.com,El joven acusado de patronear una patera de Argelia a Cabrera dice que era un pasajero más y que pagó 1.700 euros,https://www.elconfidencialdigital.com/articulo/ultima-hora/joven-acusado-patronear-patera-argelia-cabrera-dice-que-e...,El joven acusado de patronear una patera de Argelia a Cabrera dice que era un pasajero más y que pagó 1.700 euros\nS...
37359,okdiario.com,La poliomielitis y el sarampión vuelven a resurgir: los pediatras piden no bajar la cobertura vacunal,https://okdiario.com/salud/poliomielitis-sarampion-vuelven-resurgir-pediatras-piden-no-bajar-cobertura-vacunal-10766746,Fact checked\nEste artículo de OkSalud ha sido verificado para garantizar la mayor precisión y veracidad posible: se...
105604,elconfidencialdigital.com,"Rueda recrimina al Gobierno la información ""insuficiente"" sobre la llegada de migrantes y ve descoordinación",https://www.elconfidencialdigital.com/articulo/ultima-hora/rueda-recrimina-gobierno-informacion-insuficiente-llegada...,"Rueda recrimina al Gobierno la información ""insuficiente"" sobre la llegada de migrantes y ve descoordinación\nSANTIA..."
116480,vilaweb.cat,"Albert Galimany, el músic del Penedès que va triomfar al Panamà",https://www.vilaweb.cat/noticies/albert-galimany-el-music-del-penedes-que-va-triomfar-a-panama/,22.12.2023 - 21:48\n|\nActualització: 24.09.2024 - 03:53\nSalón de la Nacionalidad\nMinisteri de Governació i Justíc...
34137,lavozdegalicia.es,Galiverso somerxido: asÃ­ se recrea unha visita virtual polo fondo mariÃ±o de Galicia,https://www.lavozdegalicia.es/noticia/cultura/2023/04/05/galiverso-somerxido/00031680692820377627378.htm,A Xunta renova a súa instalación de realidade virtual propoñendo un paseo en batiscafo para coñecer o rico patrimoni...


## 11. Revisar ejemplos extraídos

In [25]:
ejemplos = df_test[df_test['texto'].notna()].copy()

for i, fila in ejemplos.head(3).iterrows():
    print('='*100)
    print('MEDIO:', fila['media_name'])
    print('TÍTULO:', fila['title'])
    print('URL:', fila['url'])
    print('-'*100)
    print(fila['texto'][:1000])
    print()

MEDIO: elconfidencialdigital.com
TÍTULO: El joven acusado de patronear una patera de Argelia a Cabrera dice que era un pasajero más y que pagó 1.700 euros
URL: https://www.elconfidencialdigital.com/articulo/ultima-hora/joven-acusado-patronear-patera-argelia-cabrera-dice-que-era-pasajero-mas-que-pago-1700-euros/20230206124258516237.html
----------------------------------------------------------------------------------------------------
El joven acusado de patronear una patera de Argelia a Cabrera dice que era un pasajero más y que pagó 1.700 euros
Sólo dos niños llevaban chaleco, según la Guardia Civil
PALMA, 6 (EUROPA PRESS)
El joven acusado de haber patroneado una patera de Argelia a Cabrera con 30 personas a bordo el pasado agosto se ha declarado inocente, este lunes en la Audiencia Provincial, asegurando que fue un pasajero más y que pagó 1.700 euros por el viaje.
La Fiscalía pide seis años de prisión para el acusado, de 21 años de edad, de un delito contra los derechos de los ciuda

In [26]:
df_filtrado["title"].sample(
    100,
    random_state=42
).tolist()

['La Guardia Civil detiene a nueve personas por favorecer el empadronamiento fraudulento de migrantes en situación irregular',
 'La llegada de migrantes a la frontera de EE UU con México, en imágenes',
 'Rescatan una patera con 85 migrantes y una persona fallecida en Gran Canaria',
 'Mueren diez migrantes tras hundirse una embarcación frente a las costas de Túnez',
 'Siete horas de angustia y espera para un rescate que deja trece muertos más en la ruta canaria de la inmigración',
 'Manuel Domínguez sostiene que la política en materia de inmigración “debe tener en Canarias el eje central de sus actuaciones”',
 'Desarticulada una mafia que vendía documentación falsa a los migrantes para salir de Canarias',
 'Tres fábricas ilegales de tabaco explotaban a refugiados ucranianos',
 'Fallece un migrante en un cayuco interceptado en Tenerife',
 'Al menos cuatro muertos y 51 desaparecidos tras el naufragio de un barco de migrantes en Túnez',
 'La falta de medios policiales impide investigar a l

In [27]:
print("VALIDACIÓN SUPERADA")
print(f"Noticias filtradas: {len(df_filtrado):,}")
print(f"Tasa extracción piloto: {exito:.2f}%")

VALIDACIÓN SUPERADA
Noticias filtradas: 22,273
Tasa extracción piloto: 91.00%


In [28]:
BATCH_SIZE = 500

for inicio in range(0, len(df_filtrado), BATCH_SIZE):

    fin = min(inicio + BATCH_SIZE, len(df_filtrado))

    bloque = df_filtrado.iloc[inicio:fin].copy()

    bloque["texto"] = bloque["url"].progress_apply(
        extraer_texto_trafilatura
    )

    modo = "w" if inicio == 0 else "a"

    bloque.to_csv(
        RUTA_SALIDA,
        mode=modo,
        header=(inicio == 0),
        index=False
    )

    print(f"Guardado bloque {inicio}-{fin}")

100%|██████████| 500/500 [09:36<00:00,  1.15s/it]


Guardado bloque 0-500


100%|██████████| 500/500 [08:22<00:00,  1.01s/it]


Guardado bloque 500-1000


100%|██████████| 500/500 [12:03<00:00,  1.45s/it]


...

Guardado bloque 21000-21500


100%|██████████| 500/500 [08:42<00:00,  1.04s/it]


Guardado bloque 21500-22000


100%|██████████| 273/273 [05:14<00:00,  1.15s/it]

Guardado bloque 22000-22273


## 12. Decisión metodológica

Antes de continuar, interpreta la tasa de extracción:

- **Más de 70%** → podemos seguir con Trafilatura.
- **Entre 40% y 70%** → se puede seguir, pero revisando medios problemáticos.
- **Menos de 40%** → conviene buscar otra estrategia antes de extraer todo.


In [29]:
df_resultado = pd.read_csv(RUTA_SALIDA)

print(df_resultado.shape)

(22273, 10)


In [30]:
exito_final = (
    df_resultado["texto"]
    .notna()
    .mean()
    * 100
)

print(f"Éxito final: {exito_final:.2f}%")

Éxito final: 92.37%


In [31]:
df_resultado["longitud_texto"] = (
    df_resultado["texto"]
    .fillna("")
    .str.len()
)

df_resultado["longitud_texto"].describe()

count    22273.000000
mean      2884.115790
std       2547.895178
min          0.000000
25%       1201.000000
50%       2430.000000
75%       3992.000000
max      71032.000000
Name: longitud_texto, dtype: float64

In [32]:
df_resultado["url"].duplicated().sum()

np.int64(0)

## 13. Extracción completa

Ejecutar esta celda solo si la prueba piloto ha funcionado bien.

Puede tardar bastante con miles de URLs.

In [ ]:
# Ejecutar solo cuando decidamos continuar

# df['texto'] = df['url'].progress_apply(extraer_texto_trafilatura)

# df['texto_extraido'] = df['texto'].notna()
# print('Tasa final de extracción:', df['texto_extraido'].mean() * 100)

# df.head()

## 14. Guardar dataset procesado

In [ ]:
# Ejecutar después de la extracción completa

# df.to_csv(RUTA_SALIDA, index=False)
# print('Archivo guardado en:', RUTA_SALIDA)

## 15. Comprobación final

In [ ]:
# df_final = pd.read_csv(RUTA_SALIDA)
# print(df_final.shape)
# df_final[['publish_date', 'media_name', 'title', 'url', 'texto']].head()